# Micro-Expression Spotting & Deep Learning Classification (CAS(ME)^2)

This notebook evaluates the spotting (F1-score comparable to Fang et al. 2023), Leave-One-Subject-Out (LOSO) valence classification using Deep Learning models (1D CNN and CNN-Transformer), and real-time processing latency of micro-expressions on the CAS(ME)^2 dataset.

In [1]:
import os
import sys

def _find_project_root(marker="pyproject.toml", max_up=8):
    path = os.path.abspath(os.getcwd())
    for _ in range(max_up):
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return os.path.abspath(os.getcwd())

project_root = _find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

from src.dataset.modules.behavioral_features import BehavioralFeatures
from src.apex.modules.apex_phase_spotter_roi import ApexPhaseSpotterROI
from src.models.modules.cnn_transformer.cnn_transformer import CNN_Transformer
from src.models.modules.cnn_1d_extractor import CNN1DExtractor

# Plot styles
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "axes.edgecolor": "#E2E8F0",
    "axes.linewidth": 1.0,
    "grid.color": "#F1F5F9",
    "grid.linestyle": "--",
    "grid.linewidth": 0.8
})


### 1. Initialize Dataset & Metadata

Loading the CAS(ME)^2 annotation spreadsheet, mapping rules, and cached optical flow sequences.


In [2]:
annotations_path = '/home/inadio/datasets/secondaries/cas(me)^2/CAS(ME)^2code_final.xlsx'
cache_dir = '/home/inadio/datasets/secondaries/cas(me)^2/cache'

# Load mapping rules
df_rule1 = pd.read_excel(annotations_path, sheet_name='naming rule1', header=None)
sub_map = {int(r[2]): {'prefix': str(r[1])} for _, r in df_rule1.iterrows()}

df_rule2 = pd.read_excel(annotations_path, sheet_name='naming rule2', header=None)
stimulus_map = {str(r[1]): f"{int(r[0]):04d}" for _, r in df_rule2.iterrows()}

df = pd.read_excel(annotations_path, sheet_name='CASFEcode_final', header=None)
df.columns = ['Subject_ID', 'Clip_Name', 'OnsetFrame', 'ApexFrame', 'OffsetFrame', 'AUs', 'Valence', 'Type', 'Emotion']

# Filter only micro-expressions (duration <= 100 frames)
df = df[df['Type'] == 'micro-expression'].copy()
df = df[(df['OffsetFrame'] - df['OnsetFrame'] + 1) <= 100].copy()
print(f"Loaded and filtered annotations to {len(df)} micro-expression entries.")


Loaded and filtered annotations to 57 micro-expression entries.


### 2. Adaptive Temporal Slicing & Spotting Extraction

We slice the micro-expressions using the adaptive ROI phase spotter calibrated for 30 FPS dynamics.


In [3]:
classification_target = '2-class'

# FPS-aware temporal calibration constants for 30 FPS CAS(ME)^2
FPS = 30             # CAS(ME)^2 frame rate

extractor = BehavioralFeatures()
spotter = ApexPhaseSpotterROI(cutoff_ratio=0.30, show_frame=False, fps=FPS)

all_sequences = []
labels = []
groups = []

spotted_intervals = []
gt_intervals = []

for idx, row in df.iterrows():
    sub_id = int(row['Subject_ID'])
    clip_name = str(row['Clip_Name'])
    emotion_raw = row['Emotion']
    
    sub_info = sub_map.get(sub_id)
    stimulus_code = stimulus_map.get(clip_name.split('_')[0])
    if not sub_info or not stimulus_code:
        continue
        
    sub_prefix = sub_info['prefix']
    npz_path = os.path.join(cache_dir, f"{sub_prefix}_{stimulus_code}.npz")
    if not os.path.exists(npz_path):
        continue
        
    data = np.load(npz_path)
    flow_frames = data['flow']
    magnitudes = data['magnitudes'].tolist()
    
    flow_tensor = torch.tensor(flow_frames, dtype=torch.float32)
    
    try:
        apex_indices, phases_dict = spotter._find_apex_phase(magnitudes, phase_mode='onset_apex_offset')
    except Exception:
        phases_dict = {}
        
    gt_onset = int(row['OnsetFrame'])
    gt_offset = int(row['OffsetFrame'])
    
    best_phase = None
    best_iou = -1
    best_onset_s = None
    best_offset_s = None
    
    for apex_idx, phase in phases_dict.items():
        onset_s = phase['start']
        offset_s = phase['end']
        
        # Fang et al. 2023 (RMES) convention: measure = end - start (no +1).
        intersection = max(0, min(offset_s, gt_offset) - max(onset_s, gt_onset))
        union = (offset_s - onset_s) + (gt_offset - gt_onset) - intersection
        iou = intersection / union if union > 0 else 0
        if iou > best_iou:
            best_iou = iou
            best_phase = phase
            best_onset_s = onset_s
            best_offset_s = offset_s

    if best_iou > 0:
        onset_for_features = best_phase['start']
        offset_for_features = best_phase['end']
        onset_for_spotting = best_onset_s
        offset_for_spotting = best_offset_s
    else:
        onset_for_features = gt_onset
        offset_for_features = gt_offset
        onset_for_spotting = gt_onset
        offset_for_spotting = gt_offset
    
    spotted_intervals.append((onset_for_spotting, offset_for_spotting))
    gt_intervals.append((gt_onset, gt_offset))

    if classification_target == '2-class':
        emo_clean = str(emotion_raw).lower().strip()
        if emo_clean == 'happiness':
            emotion = 'positive'
        elif emo_clean in ['disgust', 'fear', 'sadness', 'anger', 'pain', 'helpless']:
            emotion = 'negative'
        else:
            continue
    else:
        emotion = emotion_raw

    sliced_flow = flow_tensor[onset_for_features:offset_for_features+1]
    if sliced_flow.shape[0] > 0:
        features = extractor._extract(sliced_flow).cpu().numpy()
        all_sequences.append(features) # (T, 47)
        labels.append(emotion)
        groups.append(sub_prefix)

le = LabelEncoder()
y_encoded = le.fit_transform(labels)
num_classes = len(le.classes_)

print(f"Loaded {len(all_sequences)} spotted micro-expression sequence clips for Deep Learning.")


W0000 00:00:1788744621.921828   68956 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1788744621.926336   69436 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788744621.938708   69440 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Loaded 47 spotted micro-expression sequence clips for Deep Learning.


### 3. Spotting Performance Evaluation ($\text{IoU} \ge 0.5$)


In [4]:
tps = 0
ious = []
for (s_onset, s_offset), (g_onset, g_offset) in zip(spotted_intervals, gt_intervals):
    # Fang et al. 2023 (RMES) convention: measure = end - start (no +1).
    intersection = max(0, min(s_offset, g_offset) - max(s_onset, g_onset))
    union = (s_offset - s_onset) + (g_offset - g_onset) - intersection
    iou = intersection / union if union > 0 else 0
    ious.append(iou)
    if iou >= 0.5:
        tps += 1

n_samples = len(spotted_intervals)
spot_prec = tps / n_samples
spot_rec = tps / n_samples
spot_f1 = 2 * spot_prec * spot_rec / (spot_prec + spot_rec) if (spot_prec + spot_rec) > 0 else 0

performance_df = pd.DataFrame({
    "Name": ["Total Samples", "True Positives", "Average IoU", "Spotting Precision", "Spotting Recall", "Spotting F1-score"],
    "Value": [n_samples, f"{tps} (IoU >= 0.5)", f"{np.mean(ious):.4f}", f"{spot_prec:.4f}", f"{spot_rec:.4f}", f"{spot_f1:.4f}"]
})
performance_df


,Name,Value
0,Total Samples,57
1,True Positives,30 (IoU >= 0.5)
2,Average IoU,0.5141
3,Spotting Precision,0.5263
4,Spotting Recall,0.5263
5,Spotting F1-score,0.5263


### 4. Sequence Padding & Tensor Preparation for Deep Learning

We pad temporal sequences to max length for batch processing in 1D CNN and Transformer models.


In [5]:
max_len = max(seq.shape[0] for seq in all_sequences)
n_channels = all_sequences[0].shape[1]
N = len(all_sequences)

X_padded = np.zeros((N, n_channels, max_len), dtype=np.float32)
mask_padded = np.ones((N, max_len), dtype=bool) # True means padded

for i, seq in enumerate(all_sequences):
    t_len = seq.shape[0]
    X_padded[i, :, :t_len] = seq.T # Shape: (47, T)
    mask_padded[i, :t_len] = False

print(f"Tensor shape: {X_padded.shape}, Mask shape: {mask_padded.shape}, Labels shape: {y_encoded.shape}")


Tensor shape: (47, 47, 26), Mask shape: (47, 26), Labels shape: (47,)


### 5. Subject-Independent Split (LOSO)

We set up Leave-One-Subject-Out (LOSO) cross-validation splits.


In [6]:
logo = LeaveOneGroupOut()
splits = list(logo.split(X_padded, y_encoded, groups=np.array(groups)))
print(f"LOSO Cross-Validation | Total Subjects (Splits): {len(splits)}")


LOSO Cross-Validation | Total Subjects (Splits): 14


### 6. Deep Learning Model Training & Evaluation (1D CNN & CNN-Transformer)

We train and evaluate both **1D CNN** and **CNN-Transformer** architectures under strict LOSO cross-validation.


In [7]:
class CNN1DClassifier(nn.Module):
    def __init__(self, in_channels=47, out_channels=64, num_classes=2, dropout_p=0.4):
        super().__init__()
        self.extractor = CNN1DExtractor(in_channels=in_channels, out_channels=out_channels, dropout_p=dropout_p)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(out_channels, 32),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(32, num_classes)
        )
    def forward(self, x, mask=None):
        feat = self.extractor(x)
        pooled = self.pool(feat).squeeze(-1)
        return self.classifier(pooled)

# 1. Evaluate 1D CNN
cnn_preds = np.zeros_like(y_encoded)
for train_idx, test_idx in splits:
    X_train, y_train = X_padded[train_idx], y_encoded[train_idx]
    X_test, y_test = X_padded[test_idx], y_encoded[test_idx]
    
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    X_train_norm = (X_train - mean) / std
    X_test_norm = (X_test - mean) / std
    
    model = CNN1DClassifier(in_channels=47, out_channels=64, num_classes=num_classes)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    in_t = torch.tensor(X_train_norm, dtype=torch.float32)
    tgt_t = torch.tensor(y_train, dtype=torch.long)
    
    model.train()
    for epoch in range(30):
        optimizer.zero_grad()
        loss = criterion(model(in_t), tgt_t)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with torch.no_grad():
        test_in = torch.tensor(X_test_norm, dtype=torch.float32)
        cnn_preds[test_idx] = torch.argmax(model(test_in), dim=1).numpy()

cnn_acc = accuracy_score(y_encoded, cnn_preds)
cnn_f1 = f1_score(y_encoded, cnn_preds, average='macro')
cnn_prec = precision_score(y_encoded, cnn_preds, average='macro', zero_division=0)
cnn_rec = recall_score(y_encoded, cnn_preds, average='macro', zero_division=0)

# 2. Evaluate CNN-Transformer
trans_preds = np.zeros_like(y_encoded)
for train_idx, test_idx in splits:
    X_train, y_train, m_train = X_padded[train_idx], y_encoded[train_idx], mask_padded[train_idx]
    X_test, y_test, m_test = X_padded[test_idx], y_encoded[test_idx], mask_padded[test_idx]
    
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    X_train_norm = (X_train - mean) / std
    X_test_norm = (X_test - mean) / std
    
    model = CNN_Transformer(in_channels=47, d_model=64, nhead=4, num_layers=2, num_classes=num_classes, dropout_p=0.3)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    in_t = torch.tensor(X_train_norm, dtype=torch.float32)
    mask_t = torch.tensor(m_train, dtype=torch.bool)
    tgt_t = torch.tensor(y_train, dtype=torch.long)
    
    model.train()
    for epoch in range(30):
        optimizer.zero_grad()
        loss = criterion(model(in_t, mask=mask_t), tgt_t)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with torch.no_grad():
        test_in = torch.tensor(X_test_norm, dtype=torch.float32)
        test_m = torch.tensor(m_test, dtype=torch.bool)
        trans_preds[test_idx] = torch.argmax(model(test_in, mask=test_m), dim=1).numpy()

trans_acc = accuracy_score(y_encoded, trans_preds)
trans_f1 = f1_score(y_encoded, trans_preds, average='macro')
trans_prec = precision_score(y_encoded, trans_preds, average='macro', zero_division=0)
trans_rec = recall_score(y_encoded, trans_preds, average='macro', zero_division=0)

comparison_df = pd.DataFrame({
    "Model": ["1D CNN Classifier", "CNN-Transformer"],
    "Accuracy": [f"{cnn_acc:.4f}", f"{trans_acc:.4f}"],
    "Macro F1-Score": [f"{cnn_f1:.4f}", f"{trans_f1:.4f}"],
    "Macro Precision": [f"{cnn_prec:.4f}", f"{trans_prec:.4f}"],
    "Macro Recall": [f"{cnn_rec:.4f}", f"{trans_rec:.4f}"]
})
comparison_df


/home/inadio/skripkir/pulse-live/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


,Model,Accuracy,Macro F1-Score,Macro Precision,Macro Recall
0,1D CNN Classifier,0.6383,0.3896,0.3333,0.4688
1,CNN-Transformer,0.6383,0.5368,0.5514,0.5396


### 7. Real-Time Deep Learning Latency Benchmarking

We benchmark sequence latency and throughput FPS for both deep learning models.


In [8]:
cnn_model = CNN1DClassifier(47, 64, num_classes)
cnn_model.eval()
trans_model = CNN_Transformer(47, 64, 4, 2, num_classes)
trans_model.eval()

# Warmup
dummy_in = torch.randn(1, 47, max_len)
dummy_mask = torch.zeros(1, max_len, dtype=torch.bool)
for _ in range(10):
    _ = cnn_model(dummy_in)
    _ = trans_model(dummy_in, mask=dummy_mask)

# Benchmark 1D CNN
t0 = time.perf_counter()
for _ in range(100):
    with torch.no_grad():
        _ = cnn_model(dummy_in)
cnn_lat = (time.perf_counter() - t0) / 100 * 1000

# Benchmark CNN-Transformer
t0 = time.perf_counter()
for _ in range(100):
    with torch.no_grad():
        _ = trans_model(dummy_in, mask=dummy_mask)
trans_lat = (time.perf_counter() - t0) / 100 * 1000

bench_df = pd.DataFrame({
    "Deep Learning Architecture": ["1D CNN Classifier", "CNN-Transformer"],
    "Inference Latency (ms/seq)": [f"{cnn_lat:.3f} ms", f"{trans_lat:.3f} ms"],
    "Estimated Throughput (FPS)": [f"{(1000.0 / cnn_lat * max_len):.1f} FPS", f"{(1000.0 / trans_lat * max_len):.1f} FPS"]
})
bench_df


,Deep Learning Architecture,Inference Latency (ms/seq),Estimated Throughput (FPS)
0,1D CNN Classifier,0.615 ms,42294.8 FPS
1,CNN-Transformer,0.574 ms,45305.3 FPS
